In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import torch
from models.linear_probes import  linear_probe_tuned
from models.feature_generation import pool_features
import pandas as pd
import pickle
from pathlib import Path
import os
from models.abmil_model import abmil_classifier_tuned

In [ ]:
dir = Path(os.getcwd()).resolve().parent 
path = str(dir / "models" / "features")
X_per2 = np.load(path + "\\perch_features.npz")['features']
X_per2_pooled = X_per2
y = np.load(path + "\\perch_features.npz")['labels']
label_names = ['Type A', 'Type B', 'Type C', 'Type D', 'Echo']

In [ ]:
with open(path + "\\bat_features_mil.pkl",'rb') as f :
    loaded_features = pickle.load(f) 
X_bags = pool_features(loaded_features, windows  = False,window_pooled  = False, method  ='mean',encoder  = 'perch2')

In [ ]:
abmil_results = abmil_classifier_tuned(X_bags, y, n_split_out=5,n_split_in=5, num_trials=5,random_state=42,n_iter_search = 4)

In [ ]:
per2_results = linear_probe_tuned(X_per2_pooled, y, n_split_out=5,n_split_in=5, num_trials=5,random_state=42)

In [ ]:
ab2_results = abmil_results

In [ ]:
from evaluation.metrics import plot_abmil_vs_perch_boxplots

plot_abmil_vs_perch_boxplots(abmil_results, per2_results, label_names=label_names)

In [ ]:
from evaluation.statistical_tests import evaluate_abmil_vs_perch_statistical
evaluate_abmil_vs_perch_statistical(abmil_results, per2_results, label_names=label_names)

In [ ]:
abmil_results_feats3 = abmil_classifier_tuned(X_bags, y, n_split_out=5,n_split_in=5, num_trials=5,random_state=42,n_iter_search = 4,hybrid = True)

In [ ]:
plot_abmil_vs_perch_boxplots(abmil_results_feats3, per2_results, label_names=label_names)

In [ ]:
evaluate_abmil_vs_perch_statistical(abmil_results_feats3, per2_results, label_names=label_names)

In [ ]:
per2_results2 = linear_probe_tuned(X_per2_pooled, y, n_split_out=5,n_split_in=5, num_trials=5,random_state=42)

In [ ]:
plot_abmil_vs_perch_boxplots(abmil_results_feats3, per2_results2, label_names=label_names)

In [ ]:
def extract_essential_results(results):
    """Strip PyTorch objects, keep only numpy arrays and metrics."""
    clean = []
    for trial_result in results:
        clean.append({
            'trial': trial_result['trial'],
            'model': trial_result['model'],
            'mean_AP': trial_result['mean_AP'],
            'std_AP': trial_result['std_AP'],
            'y_true_cv': trial_result['y_true_cv'],
            'y_pred_proba_cv': trial_result['y_pred_proba_cv'],
            'oof_y_true': trial_result['oof_y_true'],
            'oof_y_pred_proba': trial_result['oof_y_pred_proba'],
            'oof_indices': trial_result['oof_indices'],
            'train_histories': trial_result['train_histories'],
            'val_histories': trial_result['val_histories'],
            # Deliberately exclude 'best_models' — contains PyTorch objects
        })
    return clean

In [ ]:
import pickle

pickle_filename = 'abmil_results.pkl'
with open(pickle_filename, 'wb') as pickle_file:
    pickle.dump(extract_essential_results(abmil_results), pickle_file)
pickle_filename = 'abmil_results_feats.pkl'
with open(pickle_filename, 'wb') as pickle_file:
    pickle.dump(extract_essential_results(abmil_results_feats3), pickle_file)

In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['TORCH_USE_CUDA_DSA'] = '1'

In [ ]:

abmil_results_ensemble = abmil_classifier_tuned(X_bags, y, n_split_out=5,n_split_in=5, num_trials=5,random_state=42,n_iter_search = 4,ensemble=True)

In [ ]:
from evaluation.metrics import plot_abmil_vs_perch_boxplots
plot_abmil_vs_perch_boxplots(abmil_results_ensemble, per2_results, label_names=label_names)

In [ ]:
import pickle
pickle_filename = 'abmil_results_ensemble.pkl'
with open(pickle_filename, 'wb') as pickle_file:
    pickle.dump(extract_essential_results(abmil_results_ensemble), pickle_file)